In [15]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import pickle
import os

In [16]:
patient_ids = ["140", "141", "247", "257", "260","313","362","404","437","446","491"]

base_path = "/Users/krisha/Projs/Neuro-brain-states/Datasets/EPIStress"

In [17]:
tasks = {
    "arithmetix_easy": "easy",
    "arithmetix_hard": "hard",
    "relaxation_video": "relaxation"
}

def read_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

all_data = []

for patient_id in patient_ids:

    for task_folder, task_label in tasks.items():

        path = os.path.join(
            base_path,
            "ES" + patient_id,
            "Lab",
            "Features",
            task_folder,
            "EEG_features_30_sec.pickle"
        )

        if not os.path.exists(path):
            print(f"Missing: {path}")
            continue

        # Original dataframe: (n windows, k features)
        df = read_pickle(path).copy()

        # Average each feature across all windows
        feature_means = df.mean(axis=0).to_frame().T

        # Add metadata
        feature_means["task_label"] = task_label
        feature_means["patient_id"] = patient_id

        all_data.append(feature_means)

        print(
            f"{patient_id} | {task_label} | "
            f"{df.shape[0]} windows -> 1 row"
        )

# Combine all patient-task rows
all_data = pd.concat(all_data, ignore_index=True)

140 | easy | 96 windows -> 1 row
140 | hard | 96 windows -> 1 row
140 | relaxation | 96 windows -> 1 row
141 | easy | 93 windows -> 1 row
141 | hard | 96 windows -> 1 row
141 | relaxation | 96 windows -> 1 row
Missing: /Users/krisha/Projs/Neuro-brain-states/Datasets/EPIStress/ES247/Lab/Features/arithmetix_easy/EEG_features_30_sec.pickle
Missing: /Users/krisha/Projs/Neuro-brain-states/Datasets/EPIStress/ES247/Lab/Features/arithmetix_hard/EEG_features_30_sec.pickle
Missing: /Users/krisha/Projs/Neuro-brain-states/Datasets/EPIStress/ES247/Lab/Features/relaxation_video/EEG_features_30_sec.pickle
257 | easy | 96 windows -> 1 row
257 | hard | 96 windows -> 1 row
257 | relaxation | 96 windows -> 1 row
260 | easy | 96 windows -> 1 row
260 | hard | 94 windows -> 1 row
260 | relaxation | 95 windows -> 1 row
313 | easy | 96 windows -> 1 row
313 | hard | 96 windows -> 1 row
313 | relaxation | 94 windows -> 1 row
362 | easy | 96 windows -> 1 row
362 | hard | 94 windows -> 1 row
362 | relaxation | 96

In [18]:

metadata_columns = [
    "task_label",
    "patient_id"
]

eeg_features = [
    col for col in all_data.columns
    if col not in metadata_columns
]

print("Number of EEG features:", len(eeg_features))
print(eeg_features[:10])

Number of EEG features: 193
['δ_TP9', 'θ_TP9', 'α_TP9', 'β_TP9', 'γ_TP9', 'δ_AF7', 'θ_AF7', 'α_AF7', 'β_AF7', 'γ_AF7']


In [19]:

def remove_redundant_wavelet_descriptors(features):

    remove_keywords = [
        "_std",
        "_rms",
        "_energy"
    ]

    selected = []

    for feature in features:

        if any(
            feature.endswith(keyword)
            for keyword in remove_keywords
        ):
            continue

        selected.append(feature)

    return selected


semantic_features = remove_redundant_wavelet_descriptors(
    eeg_features
)

print("Original features:", len(eeg_features))
print("After semantic reduction:", len(semantic_features))

print("\nRemoved:")
print(
    sorted(
        set(eeg_features) - set(semantic_features)
    )
)

Original features: 193
After semantic reduction: 133

Removed:
['AF7_a5_energy', 'AF7_a5_rms', 'AF7_a5_std', 'AF7_d3_energy', 'AF7_d3_rms', 'AF7_d3_std', 'AF7_d4_energy', 'AF7_d4_rms', 'AF7_d4_std', 'AF7_d5_energy', 'AF7_d5_rms', 'AF7_d5_std', 'AF8_a5_energy', 'AF8_a5_rms', 'AF8_a5_std', 'AF8_d3_energy', 'AF8_d3_rms', 'AF8_d3_std', 'AF8_d4_energy', 'AF8_d4_rms', 'AF8_d4_std', 'AF8_d5_energy', 'AF8_d5_rms', 'AF8_d5_std', 'TP10_a5_energy', 'TP10_a5_rms', 'TP10_a5_std', 'TP10_d3_energy', 'TP10_d3_rms', 'TP10_d3_std', 'TP10_d4_energy', 'TP10_d4_rms', 'TP10_d4_std', 'TP10_d5_energy', 'TP10_d5_rms', 'TP10_d5_std', 'TP9_a5_energy', 'TP9_a5_rms', 'TP9_a5_std', 'TP9_d3_energy', 'TP9_d3_rms', 'TP9_d3_std', 'TP9_d4_energy', 'TP9_d4_rms', 'TP9_d4_std', 'TP9_d5_energy', 'TP9_d5_rms', 'TP9_d5_std', 'avg_a5_energy', 'avg_a5_rms', 'avg_a5_std', 'avg_d3_energy', 'avg_d3_rms', 'avg_d3_std', 'avg_d4_energy', 'avg_d4_rms', 'avg_d4_std', 'avg_d5_energy', 'avg_d5_rms', 'avg_d5_std']


## Now running spearman correlation on semantic features

In [20]:

X_semantic = all_data[semantic_features]

corr_semantic = X_semantic.corr(
    method="spearman"
)

upper = corr_semantic.where(
    np.triu(
        np.ones(corr_semantic.shape),
        k=1
    ).astype(bool)
)

to_remove = [
    column
    for column in upper.columns
    if any(
        abs(upper[column]) >= 0.90
    )
]

features_after_corr = [
    f for f in semantic_features
    if f not in to_remove
]

print("Original:", len(eeg_features))
print("After semantic reduction:", len(semantic_features))
print("After correlation filtering:", len(features_after_corr))

Original: 193
After semantic reduction: 133
After correlation filtering: 87


In [21]:
from sklearn.model_selection import LeaveOneGroupOut

groups = all_data["patient_id"]

logo = LeaveOneGroupOut()

In [22]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    classification_report
)

In [23]:

X = all_data[features_after_corr].copy()

y = all_data["task_label"].copy()

groups = all_data["patient_id"].copy()

print("X shape:", X.shape)
print("Number of candidate features:", X.shape[1])
print("Number of observations:", X.shape[0])

print("\nClass distribution:")
print(y.value_counts())

print("\nPatients:")
print(groups.unique())

X shape: (30, 87)
Number of candidate features: 87
Number of observations: 30

Class distribution:
easy          10
hard          10
relaxation    10
Name: task_label, dtype: int64

Patients:
['140' '141' '257' '260' '313' '362' '404' '437' '446' '491']


In [24]:

l1_selector = SelectFromModel(
    LogisticRegression(
        penalty="l1",
        solver="saga",
        C=0.1,
        max_iter=10000,
        random_state=42
    ),
    threshold=1e-5
)

In [25]:

l1_pipeline = Pipeline([
    
    ("imputer", SimpleImputer(strategy="median")),
    
    ("scaler", StandardScaler()),
    
    ("selection", SelectFromModel(
        LogisticRegression(
            penalty="l1",
            solver="saga",
            C=0.1,
            max_iter=10000,
            random_state=42
        ),
        threshold=1e-5
    )),
    
    ("classifier", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        C=1.0,
        max_iter=10000,
        random_state=42
    ))
])

In [ ]:

selected_features_per_fold = []
fold_scores = []

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X, y, groups),
    start=1
):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    test_patient = groups.iloc[test_idx].iloc[0]

    # Train the pipeline only on the training patients
    l1_pipeline.fit(
        X_train,
        y_train
    )

    selector = l1_pipeline.named_steps["selection"]

    selected_mask = selector.get_support()

    selected_features = X.columns[selected_mask].tolist()

    selected_features_per_fold.append(
        selected_features
    )

    y_pred = l1_pipeline.predict(X_test)

    score = balanced_accuracy_score(
        y_test,
        y_pred
    )

    fold_scores.append(score)

    print(
        f"Fold {fold:2d} | "
        f"Test patient: {test_patient} | "
        f"Selected features: {len(selected_features):3d} | "
        f"Balanced accuracy: {score:.3f}"
    )

/Users/krisha/epistress_env/lib/python3.10/site-packages/sklearn/feature_selection/_base.py:122: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(


ValueError: Found array with 0 feature(s) (shape=(27, 0)) while a minimum of 1 is required by LogisticRegression.